In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

# Feature Extraction
an approach to data compression: improving storage and computational efficiency, reducing the curse of dimensionality
### Principal Component Analysis (PCA)
- identify patterns based on correlation between features to find orthogonal directions of highest variance
- project the data onto new subspace w <= dimenstions
- construct $d \times k$-dimensional transformation matrix $\mathbf W$
$$
\mathbf x = [x_1, x_2, ..., x_d],  \mathbf x \in \mathbb R^d \\
\mathbf{xW} = \mathbf z \\
\mathbf z = [z_1, z_2, ..., z_k], \mathbf z \in \mathbb R^k
$$
\
Applications: EDA, denoising stock market signals, genome data and gene expression analysis in bioinformatics \
summarized approach: 
1. standardize the dataset
2. construct covariance matrix
3. decompose into eigenvectors and eigenvalues
4. sort eigenvalues in decreasing order to rank eigenvectors
5. select top k eigenvectors
6. projection matrix
7. transform the dataset

In [ ]:
import pandas as pd
df_wine = pd.read_csv(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data',
    header=None
)
df_wine.columns = [
    'Class label',
    'Alcohol',
    'Malic acid',
    'Ash',
    'Alcalinity of ash',
    'Magnesium',
    'Total phenols',
    'Flavanoids',
    'Nonflavanoid phenols',
    'Proanthocyanins',
    'Color intensity',
    'Hue',
    'OD280/OD315 of diluted wines',
    'Proline'
]

In [ ]:
from sklearn.model_selection import train_test_split
X, y = df_wine.iloc[:, 1:].values, df_wine.iloc[:, 0].values
X_train, X_test, y_train, y_test = \
    train_test_split(X, y, test_size=0.3, stratify=y,
                     random_state=0)

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train_std = sc.fit_transform(X_train)
X_test_std = sc.transform(X_test)

Covariance between two features $x_i$ and $x_j$:
$$
\sigma_{jk} = \frac{1}{n-1} \sum_{i=1}^n (x_j^{(i)} - \mu_j)(x_k^{(i)} - \mu_k)
$$
(means are zero for standardized dataset)
$$
\implies \sigma_{jk} = \frac{1}{n-1} \sum_{i=1}^n x_j^{(i)} x_k^{(i)}
$$

In [ ]:
import numpy as np
cov_mat = np.cov(X_train_std.T)
eigenvals, eigenvecs = np.linalg.eig(cov_mat)
eigenvals

$$\text{explained variance ratio} = \frac{\lambda_j}{\sum_{j=1}^d \lambda_j}$$

In [ ]:
tot = sum(eigenvals)
var_exp = [(i/tot) for i in sorted(eigenvals, reverse=True)]
cum_var_exp = np.cumsum(var_exp)

import matplotlib.pyplot as plt
plt.bar(range(1,14), var_exp, align='center',
        label='individual explained variance')
plt.step(range(1,14), cum_var_exp, where='mid',
         label='cumulative explained variance')
plt.ylabel('explained variance ratio')
plt.xlabel('principal component index')
plt.legend(loc='center right')
plt.tight_layout()
plt.show()

In [ ]:
eigen_pairs = [(np.abs(eigenvals[i]), eigenvecs[:, i]) for i in range(len(eigenvals))]
eigen_pairs.sort(key=lambda k: k[0], reverse=True)
w = np.hstack((eigen_pairs[0][1][:, np.newaxis],
               eigen_pairs[1][1][:, np.newaxis]))
w

In [ ]:
X_train_pca = X_train_std.dot(w)
colors = ['r', 'b', 'g']
markers = ['o', 's', '^']
for l, c, m in zip(np.unique(y_train), colors, markers):
    plt.scatter(X_train_pca[y_train==l, 0],
                X_train_pca[y_train==l, 1],
                c=c, label=f'class {l}', marker=m)
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.legend(loc='best')
plt.tight_layout()
plt.show()

PCA w/ scikit-learn

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.decomposition import PCA
from utils import plot_decision_regions
pca = PCA(n_components=2)
lr = OneVsRestClassifier(LogisticRegression(solver='lbfgs',
                                            random_state=1))

X_train_pca = pca.fit_transform(X_train_std)
X_test_pca = pca.transform(X_test_std)
lr.fit(X_train_pca, y_train)
plot_decision_regions(X=X_train_pca, y=y_train, classifier=lr)
plt.xlabel('pc1')
plt.ylabel('pc2')
plt.legend(loc='best')
plt.tight_layout()
plt.show()

In [ ]:
plot_decision_regions(X=X_test_pca, y=y_test, classifier=lr)
plt.xlabel('pc1')
plt.ylabel('pc2')
plt.legend(loc='best')
plt.tight_layout()
plt.show()

In [ ]:
pca = PCA(n_components=None)
X_train_pca = pca.fit_transform(X_test_std)
pca.explained_variance_ratio_

### assessing feature contributions
loadings: how much each original feature contributes to the given principal component

In [ ]:
loadings = eigenvecs * np.sqrt(eigenvals)
fig, ax = plt.subplots()
ax.bar(range(13), loadings[:, 0], align='center')
ax.set_ylabel('Loadings for PC 1')
ax.set_xticks(range(13))
ax.set_xticklabels(df_wine.columns[1:], rotation=90)
plt.ylim([-1, 1])
plt.tight_layout()
plt.show()

In [ ]:
sklearn_loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
fig, ax = plt.subplots()
ax.bar(range(13), sklearn_loadings[:, 0], align='center')
ax.set_ylabel('loadings for pc1')
ax.set_xticks(range(13))
ax.set_xticklabels(df_wine.columns[1:], rotation=90)
plt.ylim([-1, 1])
plt.tight_layout()
plt.show()

### Linear Discriminant Analysis (LDA)
find the feature subspace that optimizes class seperability \
LDA is supervised while PCA was unsupervised \
Assumptions: data is normally distributed, classes have identical covariance matrices, examples are statistically independent \
Steps:
1. standardize $d$-dimensional dataset
2. for each class compute $d$-dimensional mean vector
3. construct between-class scatter matrix $\mathbf S_B$ and within class scatter-matrix $\mathbf S_W$
4. decompose $\mathbf S_W^{-1} \mathbf S_B$ into eigenvectors and eigenvalues
5. sort eigenvectors by eigenvalue
6. construct transformation matrix $\mathbf W$ w/ top $k$ eigenvectors
7. transform examples


**scatter matrices:** each $\mathbf m_i$ stores $\mu_m$ w/ respect to class $i$
$$
\mathbf m_i = \frac{1}{n_i} \sum_{\mathbf x \in D_i} \mathbf x_m
$$
results in:
$$
\mathbf m_i = \begin{bmatrix}
                \mu_{i,\text{alcohol}} \\
                \mu_{i,\text{malic acid}} \\
                \vdots \\
                \mu_{i,\text{proline}}
            \end{bmatrix}^T, i \in \{1, 2, 3\}
$$

In [ ]:
np.set_printoptions(precision=4)
mean_vecs = []
for label in range(1, 4):
    mean_vecs.append(np.mean(
        X_train_std[y_train==label], axis=0
    ))
    print(f'MV {label}: {mean_vecs[label - 1]}\n')


$$
\mathbf S_W = \sum_{i = 1}^c \mathbf S_i \\
\mathbf S_i = \sum_{\mathbf x \in D_i} (\mathbf x - \mathbf m_i)(\mathbf x - \mathbf m_i)^T
$$

In [ ]:
d = 13
S_W = np.zeros((d, d))
for label, mv in zip(range(1,4), mean_vecs):
    S_i = np.zeros((d, d))
    for row in X_train_std[y_train==label]:
        row, mv = row.reshape(d, 1), mv.reshape(d, 1)
        S_i += (row - mv).dot((row - mv).T)
    S_W += S_i
S_W.shape

In [ ]:
np.bincount(y_train)[1:] # we assumed that class labels were evenly distributed
# therefore, scaled each S_i before adding it to the sum

In [ ]:
S_W = np.zeros((d, d))
for label, mv in zip(range(1,4), mean_vecs):
    S_i = np.cov(X_train_std[y_train==label].T)
    S_W += S_i
S_W.shape

$$
\mathbf S_B = \sum_{i = 1}^c n_i(\mathbf m_i - \mathbf m)(\mathbf m_i - \mathbf m)^T
$$
where $\mathbf m$ is the mean across all examples

In [ ]:
m_ovr = np.mean(X_train_std, axis=0)
m_ovr = m_ovr.reshape(d,1)
S_B = np.zeros((d,d))
for i, mv in enumerate(mean_vecs):
    n = X_train_std[y_train == i + 1, :].shape[0]
    mv = mv.reshape(d, 1)
    S_B += n * (mv - m_ovr).dot((mv - m_ovr).T)
S_B.shape

In [ ]:
eigenvals, eigenvecs = np.linalg.eig(np.linalg.inv(S_W).dot(S_B))
eigen_pairs = [(np.abs(eigenvals[i]), eigenvecs[:, i]) 
               for i in range(len(eigenvecs))]
eigen_pairs = sorted(eigen_pairs, key=lambda k: k[0], reverse=True)
for ev in eigen_pairs:
    print(ev[0])

In [ ]:
tot = sum(eigenvals.real)
discr = [(i/tot) for i in sorted(eigenvals.real, reverse=True)]
cu_discr = np.cumsum(discr)
plt.bar(range(1, 14), discr, align='center', label='individual discriminability')
plt.step(range(1, 14), cu_discr, where='mid', label='cumulative discriminability')
plt.ylabel('discriminability ratio')
plt.xlabel('linear discriminants')
plt.ylim([-0.1, 1.1])
plt.legend(loc='best')
plt.tight_layout()
plt.show()

In [ ]:
w = np.hstack((eigen_pairs[0][1][:, np.newaxis].real,
               eigen_pairs[1][1][:, np.newaxis].real))
w

$$
X' = XW
$$

In [ ]:
X_train_lda = X_train_std.dot(w)
colours = ['r', 'b', 'g']
markers = ['o', 's', '^']
for l, c, m in zip(np.unique(y_train), colours, markers):
    plt.scatter(X_train_lda[y_train==l, 0],
                X_train_lda[y_train==l, 1] * (-1),
                c=c, label=f'class {l}', marker=m)
plt.xlabel('ld1')
plt.ylabel('ld2')
plt.legend(loc='best')
plt.tight_layout()
plt.show()

LDA scikit-learn

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
lda = LDA(n_components=2)
X_train_lda = lda.fit_transform(X_train_std, y_train)
lr = OneVsRestClassifier(LogisticRegression(random_state=1, solver='lbfgs'))
lr = lr.fit(X_train_lda, y_train)
plot_decision_regions(X=X_train_lda, y=y_train, classifier=lr)
plt.xlabel('ld1')
plt.ylabel('ld2')
plt.legend(loc='best')
plt.tight_layout()
plt.show()

In [ ]:
X_test_lda = lda.transform(X_test_std)
plot_decision_regions(X=X_test_lda, y=y_test, classifier=lr)
plt.xlabel('LD 1')
plt.ylabel('LD 2')
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()

### t-distributed stochastic neighbour embeddings (t-SNE)
models points based on pairwise distances in original feature space \
embeds data into lower dimensional space s.t. those distances are preserved \
meant for visualization only, as needs to transform the whole dataset, not by points

In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()
fig, ax = plt.subplots(1, 4)
for i in range(4):
    ax[i].imshow(digits.images[i], cmap='Greys')
plt.show()
digits.data.shape

In [ ]:
from sklearn.manifold import TSNE
y_digits = digits.target
X_digits = digits.data
tsne = TSNE(n_components=2, init='pca',
            random_state=123)
X_digits_tsne = tsne.fit_transform(X_digits)

In [ ]:
import matplotlib.patheffects as PathEffects
def plot_projection(x, colours):
    f = plt.figure(figsize=(8, 8))
    ax = plt.subplot(aspect='equal')
    for i in range(10):
        plt.scatter(x[colours == i, 0],
                    x[colours == i, 1])

    for i in range(10):
        xtext, ytext = np.median(x[colours == i, :], axis=0)
        txt = ax.text(xtext, ytext, str(i), fontsize=24)
        txt.set_path_effects([
            PathEffects.Stroke(linewidth=5, foreground='w'),
            PathEffects.Normal()
        ])

plot_projection(X_digits_tsne, y_digits)
plt.show()